# NLP — Text Classification

Progressive workflow: bag-of-words -> TF-IDF + linear models -> embeddings.
Always establish a strong linear baseline before reaching for transformers.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. TF-IDF features — what does the model actually learn?

TF-IDF reweights raw counts by inverse document frequency — common words
(``the``, ``is``) get down-weighted; discriminative words get amplified.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

# Overlapping computer topics -> harder -> LR/SVC beat NB
CATS = ["comp.graphics", "comp.os.ms-windows.misc",
        "comp.sys.ibm.pc.hardware", "comp.sys.mac.hardware"]
train = fetch_20newsgroups(subset="train", categories=CATS,
                            remove=("headers","footers","quotes"))
test  = fetch_20newsgroups(subset="test",  categories=CATS,
                            remove=("headers","footers","quotes"))

models = {
    "Naive Bayes": Pipeline([("v", TfidfVectorizer(max_features=30_000)),
                              ("m", MultinomialNB())]),
    "Logistic Reg": Pipeline([("v", TfidfVectorizer(max_features=30_000, ngram_range=(1,2))),
                               ("m", LogisticRegression(C=5, max_iter=1000, random_state=42))]),
    "LinearSVC":   Pipeline([("v", TfidfVectorizer(max_features=30_000, ngram_range=(1,2))),
                              ("m", LinearSVC(C=1.0, random_state=42))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(train.data, train.target)
    preds = pipe.predict(test.data)
    from sklearn.metrics import f1_score, accuracy_score
    results[name] = {
        "f1":  f1_score(test.target, preds, average="macro"),
        "acc": accuracy_score(test.target, preds),
        "pipe": pipe, "preds": preds,
    }

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
names = list(results.keys())
f1s   = [results[n]["f1"]  for n in names]
accs  = [results[n]["acc"] for n in names]
x = np.arange(len(names))
w = 0.35
b1 = ax.bar(x - w/2, [f*100 for f in f1s],  w, color=C[0], label="Macro F1")
b2 = ax.bar(x + w/2, [a*100 for a in accs], w, color=C[2], label="Accuracy")
ax.bar_label(b1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.1f%%", padding=3, fontsize=9)
ax.set(xticks=x, xticklabels=names, ylabel="Score (%)", ylim=(50,100),
       title="Model comparison on overlapping comp.* categories
(LinearSVC beats NB on ambiguous text)")
ax.legend()

# Confusion matrix for best model
best_name = max(results, key=lambda n: results[n]["f1"])
cm = confusion_matrix(test.target, results[best_name]["preds"])
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
im = axes[1].imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
axes[1].set(xticks=range(4), yticks=range(4),
            xticklabels=[c.split(".")[-1] for c in CATS],
            yticklabels=[c.split(".")[-1] for c in CATS],
            xlabel="Predicted", ylabel="True",
            title=f"Confusion matrix - {best_name}
(% of true class, diagonal = correct)")
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f"{cm_pct[i,j]:.0f}%", ha="center", va="center",
                     color="white" if cm_pct[i,j]>50 else "black", fontsize=9)
plt.colorbar(im, ax=axes[1], label="% of true class")
plt.tight_layout()
plt.show()

## 2. Top discriminative features per class

TF-IDF + logistic regression is interpretable — the coefficients directly
show which words the model associates with each class.

In [ ]:
lr_pipe = models["LinearSVC"]
# Use LogisticRegression for coefficient access
lr_coef_pipe = Pipeline([("v", TfidfVectorizer(max_features=30_000, ngram_range=(1,2))),
                          ("m", LogisticRegression(C=5, max_iter=1000, random_state=42))])
lr_coef_pipe.fit(train.data, train.target)

vocab = lr_coef_pipe.named_steps["v"].get_feature_names_out()
coef  = lr_coef_pipe.named_steps["m"].coef_
short_cats = [c.split(".")[-1] for c in CATS]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
n_top = 12
for ax, cls_idx, cat in zip(axes, range(4), short_cats):
    top_idx = np.argsort(coef[cls_idx])[-n_top:][::-1]
    top_w   = coef[cls_idx][top_idx]
    colors  = [C[0] if w > 0 else C[1] for w in top_w]
    ax.barh(range(n_top), top_w[::-1], color=colors[::-1])
    ax.set(yticks=range(n_top), yticklabels=vocab[top_idx[::-1]],
           title=f"{cat}", xlabel="Coefficient")
    ax.invert_yaxis()
fig.suptitle("Top discriminative words per class (Logistic Regression)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Semantic similarity with TF-IDF cosine

A lightweight semantic search without any embeddings model — useful
as a baseline before adding dense retrieval.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Small illustrative corpus
corpus = [
    "machine learning fraud detection credit card anomaly",
    "neural network deep learning classification banking",
    "natural language processing text classification sentiment",
    "transformer BERT fine-tuning NLP tasks",
    "survival analysis customer churn time-to-event censoring",
    "A/B testing causal inference experiment randomisation",
    "recommender system collaborative filtering matrix factorisation",
    "gradient boosting decision tree ensemble feature importance",
]
queries = [
    "detecting fraud with ML in finance",
    "NLP models for text tasks",
    "customer retention analysis",
]

vec      = TfidfVectorizer(ngram_range=(1,2)).fit(corpus)
doc_vecs = vec.transform(corpus)
q_vecs   = vec.transform(queries)
sims     = cosine_similarity(q_vecs, doc_vecs)

fig, axes = plt.subplots(1, len(queries), figsize=(15, 4))
short_docs = [d[:35]+"..." for d in corpus]
for ax, query, sim_row in zip(axes, queries, sims):
    order = np.argsort(sim_row)[::-1]
    bars  = ax.barh(range(len(corpus)), sim_row[order], color=C[0], alpha=0.8)
    ax.set(yticks=range(len(corpus)),
           yticklabels=[short_docs[i] for i in order],
           xlabel="Cosine similarity",
           title=f'Query: "{query[:30]}..."')
    ax.invert_yaxis()
    # Highlight top-1
    ax.get_children()[0].set_color(C[1])
fig.suptitle("TF-IDF cosine similarity: corpus retrieval results", fontsize=12)
plt.tight_layout()
plt.show()